In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import japanize_matplotlib 

In [2]:
# データのパス
path_kinzoku = r"C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted\勤続年数・超過実労働時間数・所定内給与額_時系列_共通年.csv"
path_tosan = r"C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted\産業別年間倒産件数表_共通年.csv"
path_shitsugyosha = r"C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted\失業者 - シート1_共通年.csv"

# データの読み込み
df_kinzoku = pd.read_csv(path_kinzoku, encoding='utf-8')
df_tosan = pd.read_csv(path_tosan, encoding='utf-8')
df_shitsugyosha = pd.read_csv(path_shitsugyosha, encoding='utf-8')

In [11]:
kinzoku_map = {
    'Ｄ建設業': '建設業', 'Ｅ製造業': '製造業', 'Ｇ情報通信業': '情報通信業', 
    'Ｉ卸売業，小売業': '卸売業，小売業', 'Ｊ金融業，保険業': '金融・保険業', 
    'Ｋ不動産業，物品賃貸業': '不動産業', 'Ｈ運輸業，郵便業': '運輸業', 
    '(民＋公)Ｈ運輸業，郵便業': '運輸業', 'Ｃ鉱業，採石業，砂利採取業': '農・林・漁・鉱業', 
    'Ｏ教育，学習支援業': 'サービス業他', 'Ｎ生活関連サービス業，娯楽業': 'サービス業他', 
    'Ｐ医療，福祉': 'サービス業他', 'Ｌ学術研究，専門・技術サービス業': 'サービス業他', 
    'Ｍ宿泊業，飲食サービス業': 'サービス業他', 'Ｑ複合サービス事業': 'サービス業他', 
    'Ｆ電気・ガス・熱供給・水道業': 'サービス業他', '(民＋公)Ｆ電気・ガス・熱供給・水道業': 'サービス業他', 
    'Ｒサービス業(他に分類されないもの)': 'サービス業他',
    '産業計': '産業合計',
    '(民＋公)産業計': '産業_民+公'
}

tosan_map = {
    '農・林・漁・鉱業': '農・林・漁・鉱業', '建設業': '建設業', '製造業': '製造業', 
    '卸売業': '卸売業，小売業', '小売業': '卸売業，小売業', '金融・保険業': '金融・保険業', 
    '不動産業': '不動産業', '運輸業': '運輸業', '情報通信業': '情報通信業', 'サービス業他': 'サービス業他',
    '合計': '産業合計'
}

shitsugyosha_map = {'建設業': '建設業', '製造業': '製造業', '情報通信業': '情報通信業', '卸売業，小売業': '卸売業，小売業', '運輸業，郵便業': '運輸業', '金融業，保険業': '金融・保険業', '不動産業，物品賃貸業': '不動産業', '漁業': '農・林・漁・鉱業', '鉱業，採石業，砂利採取業': '農・林・漁・鉱業', '電気・ガス・熱供給・水道業': 'サービス業他', '学術研究，専門・技術サービス業': 'サービス業他', '宿泊業，飲食サービス業': 'サービス業他', '生活関連サービス業，娯楽業': 'サービス業他', '教育，学習支援業': 'サービス業他', '医療，福祉': 'サービス業他', '複合サービス事業': 'サービス業他', 'サービス業(他に分類されないもの)': 'サービス業他', '公務(他に分類されるものを除く)': 'サービス業他'}

In [14]:
df_kinzoku['共通産業列'] = df_kinzoku['産業'].map(kinzoku_map)

In [15]:
# マッピングされなかった不要な行を除外して新しいデータフレームとして確定
df_kinzoku_fixed = df_kinzoku.dropna(subset=['共通産業列']).copy()

# 主要な数値列のクレンジング（カンマやハイフンを除去して数値型に変換）
for col in ['10人_勤続年数_年', '10人_所定内給与額_千円', '10人_労働者数_千円']:
    df_kinzoku_fixed[col] = pd.to_numeric(df_kinzoku_fixed[col].astype(str).str.replace(',', '').str.replace('-', ''), errors='coerce')

In [18]:
id_vars = ['年_共通']
# 「年」や「出典」などの不要な列を除いた、産業名（および合計）の列を抽出
value_vars = [col for col in df_tosan.columns if col not in id_vars + ['年', '出典']]

df_tosan_melt = df_tosan.melt(id_vars=id_vars, value_vars=value_vars, var_name='元の産業', value_name='倒産件数')
df_tosan_melt['倒産件数'] = df_tosan_melt['倒産件数'].astype(str).str.replace(',', '').astype(float)

# 先ほど定義した tosan_map を使って「共通産業列」を作成（ここで「合計」が「産業合計」になります）
df_tosan_melt['共通産業列'] = df_tosan_melt['元の産業'].map(tosan_map)
df_tosan_fixed = df_tosan_melt.groupby(['年_共通', '共通産業列'], as_index=False)['倒産件数'].sum()

In [20]:
# 改行コードの削除と、カッコの全角半角を統一
df_shitsugyosha['クレンジング済'] = df_shitsugyosha['前職の産業'].astype(str).str.replace('\n', '').str.replace('（', '(').str.replace('）', ')')

# 先ほど定義した shitsugyosha_map を使って「共通産業列」を作成
df_shitsugyosha['共通産業列'] = df_shitsugyosha['クレンジング済'].map(shitsugyosha_map)

# 職業名など、産業にマッピングされなかった不要な行を除外
df_shitsugyosha_filtered = df_shitsugyosha.dropna(subset=['共通産業列']).copy()

# 失業者数（文字列型）を数値型に変換
for col in ['男女計（失業者数）', '男（失業者数）', '女（失業者数）']:
    df_shitsugyosha_filtered[col] = df_shitsugyosha_filtered[col].astype(str).str.replace(',', '').str.replace('-', '0').str.strip()
    df_shitsugyosha_filtered[col] = pd.to_numeric(df_shitsugyosha_filtered[col], errors='coerce').fillna(0)

# 複数の元の産業が1つの共通産業に統合されたため、年と共通産業列でグループ化して合計
df_shitsugyosha_fixed = df_shitsugyosha_filtered.groupby(['年_共通', '共通産業列'], as_index=False)[['男女計（失業者数）', '男（失業者数）', '女（失業者数）']].sum()

# --- 各産業の合計値を計算して「産業合計」の行を自作して追加 ---
df_shitsugyosha_total = df_shitsugyosha_fixed.groupby('年_共通', as_index=False)[['男女計（失業者数）', '男（失業者数）', '女（失業者数）']].sum()
df_shitsugyosha_total['共通産業列'] = '産業合計'

# 自作した合計行を元のデータフレームに結合
df_shitsugyosha_fixed = pd.concat([df_shitsugyosha_fixed, df_shitsugyosha_total], ignore_index=True)

In [21]:
print("各データフレームの「共通産業列」の最終確認")
print("1. 勤続年数データ (df_kinzoku_fixed):\n", sorted(df_kinzoku_fixed['共通産業列'].unique()))
print("\n2. 倒産件数データ (df_tosan_fixed):\n", sorted(df_tosan_fixed['共通産業列'].unique()))
print("\n3. 失業者データ (df_shitsugyosha_fixed):\n", sorted(df_shitsugyosha_fixed['共通産業列'].unique()))

各データフレームの「共通産業列」の最終確認
1. 勤続年数データ (df_kinzoku_fixed):
 ['サービス業他', '不動産業', '卸売業，小売業', '建設業', '情報通信業', '産業_民+公', '産業合計', '製造業', '農・林・漁・鉱業', '運輸業', '金融・保険業']

2. 倒産件数データ (df_tosan_fixed):
 ['サービス業他', '不動産業', '卸売業，小売業', '建設業', '情報通信業', '産業合計', '製造業', '農・林・漁・鉱業', '運輸業', '金融・保険業']

3. 失業者データ (df_shitsugyosha_fixed):
 ['サービス業他', '不動産業', '卸売業，小売業', '建設業', '情報通信業', '産業合計', '製造業', '農・林・漁・鉱業', '運輸業', '金融・保険業']


In [24]:
df_kinzoku_fixed.head()

,区 分,産業,年度,10人_勤続年数_年,10人_超過実労働時間数_時間,10人_所定内給与額_千円,10人_労働者数_千円,"1,000人以上_勤続年数_年","1,000人以上_超過実労働時間数_時間","1,000人以上_所定内給与額_千円",...,100～999人_勤続年数_年,100～999人_超過実労働時間数_時間,100～999人_所定内給与額_千円,100～999人_労働者数_千円,10～99人勤続年数_年,10～99人_超過実労働時間数_時間,10～99人_所定内給与額_千円,10～99人_労働者数_千円,年_共通,共通産業列
0,男 女 計\n学 歴 計,産業計,令和七年,12.7,11,340.6,2892371.0,14.1,13,385.1,...,12.4,11,326.2,1051660,11.5,9,305.6,842023,2025,産業合計
1,３５～３９歳,Ｏ教育，学習支援業,令和七年,8.1,8,401.5,4275.0,7.8,9,439.4,...,8.2,7,393.2,1681,8.6,9,337.3,825,2025,サービス業他
2,３０～３４歳,Ｏ教育，学習支援業,令和七年,5.3,9,348.0,3681.0,4.8,10,377,...,5.4,7,338,1371,6,11,307.6,767,2025,サービス業他
3,２５～２９歳,Ｏ教育，学習支援業,令和七年,3.3,10,290.6,2807.0,3.3,13,307.4,...,3.3,8,294,1152,3.5,12,262.9,713,2025,サービス業他
4,２０～２４歳,Ｏ教育，学習支援業,令和七年,1.4,8,244.1,1263.0,1.3,10,247.2,...,1.3,6,248.5,521,1.7,9,233.6,336,2025,サービス業他


In [25]:
df_tosan_fixed.head()

,年_共通,共通産業列,倒産件数
0,2014,サービス業他,2441.0
1,2014,不動産業,340.0
2,2014,卸売業，小売業,2639.0
3,2014,建設業,1965.0
4,2014,情報通信業,394.0


In [26]:
df_shitsugyosha_fixed.head()

,年_共通,共通産業列,男女計（失業者数）,男（失業者数）,女（失業者数）
0,2014,サービス業他,37,13,25
1,2014,不動産業,1,1,0
2,2014,卸売業，小売業,19,9,11
3,2014,建設業,7,6,1
4,2014,情報通信業,4,2,2


In [28]:
# 保存先のフォルダパスを指定（raw文字列として定義）
output_dir = r"C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted_update_1"

# 指定されたフォルダが存在しない場合は自動的に作成
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"フォルダを作成しました: {output_dir}")

# --- 各データフレームを名前をつけて保存 ---

# 1. 勤続年数データ
file_kinzoku = os.path.join(output_dir, "勤続年数・超過実労働時間数・所定内給与額_時系列_共通年_統一.csv")
df_kinzoku_fixed.to_csv(file_kinzoku, index=False, encoding='utf-8-sig')
print(f"保存完了: {file_kinzoku}")

# 2. 倒産件数データ
file_tosan = os.path.join(output_dir, "産業別年間倒産件数表_共通年_統一.csv")
df_tosan_fixed.to_csv(file_tosan, index=False, encoding='utf-8-sig')
print(f"保存完了: {file_tosan}")

# 3. 失業者データ
file_shitsugyosha = os.path.join(output_dir, "失業者 - シート1_共通年_統一.csv")
df_shitsugyosha_fixed.to_csv(file_shitsugyosha, index=False, encoding='utf-8-sig')
print(f"保存完了: {file_shitsugyosha}")

print("\nすべてのデータフレームの保存が正常に終了しました！")

フォルダを作成しました: C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted_update_1
保存完了: C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted_update_1\勤続年数・超過実労働時間数・所定内給与額_時系列_共通年_統一.csv
保存完了: C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted_update_1\産業別年間倒産件数表_共通年_統一.csv
保存完了: C:\GitHub\Data-Analysis_competition\SSDSE\02_industry_labor_economics\Data\covert_use_fileted_update_1\失業者 - シート1_共通年_統一.csv

すべてのデータフレームの保存が正常に終了しました！
